# 5 · Saturation Forecasting Dashboard

This notebook implements the interactive visualization dashboard for the **Geospatial Repletion & Saturation Modelling** project.
It loads the trained GBT forecasting model, reads the aggregated infrastructure summaries, generates saturation forecasts for future time steps, and renders an interactive Leaflet/Folium map showing dynamic city bottlenecks.

In [1]:
import os
import sys
import math
from pathlib import Path

import folium
from folium.plugins import TimestampedGeoJson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressionModel

# Find project root robustly by looking for 'src' folder upward
current = Path.cwd()
while current.name and not (current / "src").exists():
    current = current.parent
PROJECT_ROOT = current
sys.path.insert(0, str(PROJECT_ROOT.resolve()))

# Setup native Hadoop binaries on Windows
from src.step_08_bootstrapping import setup_winutils
setup_winutils(PROJECT_ROOT)

# Initialize local Spark Session
spark = (
    SparkSession.builder
    .appName("Saturation-Forecasting-Dashboard")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "10")
    .getOrCreate()
)

print(f"SparkSession started successfully. Spark version: {spark.version}")

2026-07-08 16:00:21,256 - INFO - Hadoop environment path configuration active: HADOOP_HOME=C:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling\data\winutils


SparkSession started successfully. Spark version: 3.5.8


## 5.1 Load Data and Model

We load Vienna's district boundaries and the trained GBT forecasting model from storage.

In [2]:
import json

# Load districts to center the map
districts_path = PROJECT_ROOT / "data" / "spatial" / "vienna_districts.geojson"
with open(districts_path, "r", encoding="utf-8") as f:
    districts_geojson = json.load(f)

# Load trained GBT model weights
model_path = PROJECT_ROOT / "models" / "gbt_saturation_forecaster"
gbt_model = GBTRegressionModel.load(str(model_path))
print(f"Loaded GBT Saturation Model from: {model_path}")

Loaded GBT Saturation Model from: C:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling\models\gbt_saturation_forecaster


## 5.2 Build Feature DataFrame for Future Predictions

We read the aggregate infrastructure count file and run the same feature engineering window transformations to prepare features for prediction.

In [3]:
data_path = PROJECT_ROOT / "data" / "geospatial_output" / "infrastructure_activity_counts.csv"
df_raw = spark.read.csv(str(data_path), header=True, inferSchema=True)

# Capacity scaling mapping
df_capped = df_raw.withColumn(
    "max_capacity",
    F.when(F.col("infrastructure_type") == "bike_path", 3000.0)
     .when(F.col("infrastructure_type") == "pedestrian_zone", 600.0)
     .otherwise(1000.0)
)

w_seg = Window.partitionBy("infrastructure_label", "infrastructure_type").orderBy("time_window_index")
w_roll = w_seg.rowsBetween(-3, -1)

df_features = df_capped

# Lags
for lag_idx in range(1, 7):
    df_features = df_features.withColumn(f"count_lag_{lag_idx}", F.lag("count", lag_idx).over(w_seg))

# Rolling stats
df_features = df_features.withColumn("count_rolling_mean_3", F.avg("count").over(w_roll))
df_features = df_features.withColumn("count_rolling_std_3", F.stddev("count").over(w_roll))

# Delta
df_features = df_features.withColumn("count_delta", F.col("count") - F.col("count_lag_1"))

# Remove null values created by lag windows
df_clean = df_features.na.drop()

# Encode categories
indexer = StringIndexer(inputCol="infrastructure_type", outputCol="infra_type_idx", handleInvalid="keep")
df_indexed = indexer.fit(df_clean).transform(df_clean)

# Assemble features vector
feature_cols = [
    "infra_type_idx", "count",
    "count_lag_1", "count_lag_2", "count_lag_3",
    "count_lag_4", "count_lag_5", "count_lag_6",
    "count_rolling_mean_3", "count_rolling_std_3", "count_delta"
]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
ml_df = assembler.transform(df_indexed)

print("Feature matrix ready for forecasting.")

Feature matrix ready for forecasting.


## 5.3 Generate Saturation Predictions

We use our loaded model to forecast future saturation indexes.

In [4]:
predictions = gbt_model.transform(ml_df)

# Extract coordinates/features back to drive for Folium visualization
# Select necessary fields only to preserve memory
pred_rows = (
    predictions.select(
        "infrastructure_label",
        "infrastructure_type",
        "time_window_index",
        "count",
        "max_capacity",
        "prediction"
    )
    .collect()
)

print(f"Generated forecasts for {len(pred_rows)} segment-windows.")

Generated forecasts for 2369 segment-windows.


## 5.4 Build Interactive Saturation Map

We map the forecasted saturation index over time onto Vienna's infrastructure. 
We color code segments as follows:
- **Green**: Saturation Index < 0.3 (Clear)
- **Yellow**: Saturation Index 0.3 - 0.7 (Moderate load)
- **Red**: Saturation Index >= 0.7 (Overloaded / high risk of bottleneck)

In [5]:
# Load infrastructure segment shapes to match labels
ped_zones_path = PROJECT_ROOT / "data" / "spatial" / "vienna_pedestrian_zones.geojson"
bike_paths_path = PROJECT_ROOT / "data" / "spatial" / "vienna_bike_paths.geojson"

with open(ped_zones_path, "r", encoding="utf-8") as f:
    ped_geojson = json.load(f)
with open(bike_paths_path, "r", encoding="utf-8") as f:
    bike_geojson = json.load(f)

# Index geometry coordinates by segment labels for quick mapping
geoms = {}
for feature in ped_geojson["features"]:
    label = feature["properties"].get("ADRESSE", "ped_zone")
    geoms[("pedestrian_zone", label)] = feature["geometry"]

for feature in bike_geojson["features"]:
    label = feature["properties"].get("GIP_STRNAM", "bike_path")
    geoms[("bike_path", label)] = feature["geometry"]

# Create base Folium Map centered on Vienna
vienna_map = folium.Map(location=[48.2082, 16.3738], zoom_start=13, tiles="CartoDB dark_matter")

# Compile Timestamped GeoJSON features representing predicted changes over time
time_features = []
base_time = 1700000000000  # BASE_TIMESTAMP_MS

for row in pred_rows:
    infra_type = row.infrastructure_type
    label = row.infrastructure_label
    win_idx = row.time_window_index
    count = row["count"]
    cap = row.max_capacity
    pred_sat = max(0.0, min(1.2, float(row.prediction)))  # Clamp saturation between 0 and 1.2

    geom = geoms.get((infra_type, label))
    if geom is None:
        continue

    # Determine segment color based on predicted saturation
    if pred_sat >= 0.7:
        color = "#ff0000"  # Red
    elif pred_sat >= 0.3:
        color = "#ffff00"  # Yellow
    else:
        color = "#00ff00"  # Green

    # Construct ISO8601 time string corresponding to this window
    time_ms = base_time + win_idx * 60 * 1000  # 1-minute windows
    import datetime
    time_str = datetime.datetime.fromtimestamp(time_ms / 1000.0, datetime.UTC).isoformat()

    feat = {
        "type": "Feature",
        "geometry": geom,
        "properties": {
            "time": time_str,
            "style": {
                "color": color,
                "weight": 5 if infra_type == "bike_path" else 2,
                "fillColor": color,
                "fillOpacity": 0.6 if infra_type == "pedestrian_zone" else 0.8
            },
            "popup": f"<b>Segment:</b> {label}<br/><b>Type:</b> {infra_type}<br/><b>Predicted Saturation:</b> {pred_sat:.2%}<br/><b>Forecasted Density:</b> {int(count)}/hr"
        }
    }
    time_features.append(feat)

# Add Slider control overlay
TimestampedGeoJson(
    {
        "type": "FeatureCollection",
        "features": time_features
    },
    period="PT1M",
    add_last_point=True,
    auto_play=False,
    loop=False,
    max_speed=1,
    loop_button=True,
    date_options="YYYY-MM-DD HH:mm",
    time_slider_drag_update=True
).add_to(vienna_map)

# Export map HTML file
output_dir = PROJECT_ROOT / "data" / "geospatial_output"
output_dir.mkdir(parents=True, exist_ok=True)
map_path = output_dir / "saturation_forecast_map.html"
vienna_map.save(str(map_path))
print(f"Successfully generated interactive Saturation Forecast Map at: {map_path}")

Successfully generated interactive Saturation Forecast Map at: C:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling\data\geospatial_output\saturation_forecast_map.html
